# Teste Final do Pipeline Multimodal com LangGraph + AWS S3

Este notebook realiza a validação completa do pipeline multimodal desenvolvido para apoio à triagem clínica preventiva em saúde feminina.

O fluxo integra:
- upload de arquivos multimodais no Amazon S3;
- download automático pelo LangGraph;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa com LLM.

O pipeline utiliza:
- AWS S3;
- AWS Rekognition;
- YOLOv8;
- OpenCV;
- Librosa;
- LangGraph;
- RAG;
- Mistral via Ollama.

Os testes utilizam arquivos da base pública RAVDESS, composta por atores e emoções simuladas, evitando uso de dados reais de pacientes.

A análise possui finalidade exclusivamente educacional e de apoio à triagem preventiva.

In [27]:
import sys
print(sys.executable)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Scripts\python.exe


In [28]:
!{sys.executable} -m pip install moviepy imageio imageio-ffmpeg


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
!{sys.executable} -m pip show moviepy

Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Lib\site-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [30]:
from moviepy import VideoFileClip

print("MoviePy funcionando.")

MoviePy funcionando.


In [31]:
import sys
sys.path.append("..")

import os
import boto3

from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from src.workflows.langgraph_flow import (
    build_medical_assistant_graph
)

from src.llm.ollama_client import (
    get_llm
)

from src.rag.vector_store import (
    load_vector_store
)

from src.rag.retriever import (
    get_retriever,
    retrieve_context
)

from src.multimodal.media_utils import (
    extract_audio_from_video
)

In [32]:
llm = get_llm(
    model_name="mistral",
    temperature=0.2
)

vector_store = load_vector_store(
    "../data/vector_store"
)

retriever = get_retriever(
    vector_store,
    k=3
)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

print("LangGraph compilado com sucesso.")

Base vetorial carregada de: ../data/vector_store
LangGraph compilado com sucesso.


### CONFIGURAÇÃO AWS
Nesta etapa são carregadas as variáveis de ambiente e a configuração do Amazon S3 utilizada para armazenamento temporário dos arquivos multimodais.

In [33]:
load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

print("Bucket:", bucket)
print("Region:", region)

Bucket: postechfase4
Region: us-east-1


In [34]:
s3 = boto3.client(
    "s3",
    region_name=region
)

print("Cliente S3 criado com sucesso.")

Cliente S3 criado com sucesso.



### ESCOLHA DO CENÁRIO

Nesta etapa é selecionado o cenário multimodal utilizado para teste.

Os arquivos pertencem à base pública RAVDESS e representam emoções simuladas por atores.

Cenários sugeridos:
- calm
- sad
- fearful
- angry

In [35]:
video = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\video_calm.mp4"

print("Vídeo existe?", os.path.exists(video))
print(video)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\video_calm.mp4


### CÉLULA 8 - EXTRAÇÃO DO ÁUDIO

O áudio é extraído automaticamente do vídeo para permitir análise acústica e textual da fala.

In [36]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_calm_audio_female.wav"
)

extract_audio_from_video(
    video_path=str(video),
    output_audio_path=str(audio_output)
)

print("Áudio extraído com sucesso.")

Áudio extraído com sucesso.


In [37]:

print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_calm_audio_female.wav


### CÉLULA 11 - UPLOAD S3

Nesta etapa os arquivos multimodais são enviados para o Amazon S3.

O pipeline LangGraph realizará o download automático durante a execução.

In [38]:
video_s3_key = (
    "inputs/videos/video_calm.mp4"
)

audio_s3_key = (
    "inputs/audios/extract_calm_audio_female.wav"
)

s3.upload_file(
    str(video),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [39]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [40]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/video_calm.mp4 - 4534053 bytes
inputs/audios/extract_calm_audio_female.wav - 629826 bytes


### CÉLULA 16 - BUILD LANGGRAPH

Nesta etapa o LangGraph é compilado para execução do pipeline multimodal integrado.

In [42]:
app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever
)

print("LangGraph compilado com sucesso.")

LangGraph compilado com sucesso.


### CÉLULA 18 - CRIAÇÃO DO ESTADO

O estado contém:
- pergunta do usuário;
- chaves dos arquivos multimodais no S3;
- idioma do áudio;
- identificação do cenário.

In [43]:
state = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "RAVDESS_FEARFUL_01"
}

### CÉLULA 20 - EXECUÇÃO DO PIPELINE

O pipeline executa automaticamente:
- download dos arquivos do S3;
- análise de vídeo;
- análise de áudio;
- fusão multimodal;
- geração de resposta;
- alerta preventivo.

In [44]:
result = app.invoke(state)

print("Pipeline executado com sucesso.")

Pipeline executado com sucesso.


### CÉLULA 22 - RESULTADO DO VÍDEO

Nesta etapa é exibido o resultado da análise visual.

In [45]:
pprint(result["video_result"])

{'flags': ['person_detected', 'face_not_visible'],
 'interpretation': ['Foi detectada presença humana, mas não houve detecção '
                    'facial suficiente para análise emocional.'],
 'limitations': ['A análise facial indica apenas expressões aparentes, não '
                 'estado psicológico real.',
                 'O sistema não realiza diagnóstico médico ou psicológico.',
                 'Os sinais visuais devem ser interpretados apenas como apoio '
                 'à triagem.',
                 'Iluminação, ângulo da câmera, qualidade do vídeo e oclusões '
                 'podem afetar os resultados.',
                 'O YOLOv8 utilizado detecta presença humana, mas não '
                 'substitui avaliação clínica especializada.'],
 'metadata': {'analysis_strategy': 'frame_sampling_every_5_seconds',
              'cloud_service': 'AWS Rekognition',
              'dominant_emotions': {'UNKNOWN': 1},
              'duration_seconds': 3.57,
              'emotion

In [46]:
for item in result["video_result"].get(
    "interpretation",
    []
):
    print("-", item)

- Foi detectada presença humana, mas não houve detecção facial suficiente para análise emocional.


### CÉLULA 25 - RESULTADO DO ÁUDIO

Nesta etapa é exibido o resultado da análise acústica e textual do áudio.

In [47]:
pprint(result["audio_result"])

{'detected_categories': {},
 'flags': ['voice_instability',
           'elevated_voice_tension',
           'speech_hesitation',
           'low_voice_energy'],
 'interpretation': 'O perfil vocal estimado é compatível com voz feminina. '
                   'Foram observadas oscilações vocais compatíveis com '
                   'instabilidade emocional ou tensão. Foram identificados '
                   'sinais de hesitação na fala. A análise acústica '
                   'identificou possível tensão vocal. A intensidade vocal '
                   'reduzida pode estar associada a fadiga ou cansaço. A '
                   'intensidade vocal identificada foi classificada como '
                   'baixa. A elevada variação do pitch pode indicar agitação '
                   'ou ansiedade. A presença de pausas frequentes pode indicar '
                   'insegurança ou desconforto emocional. Os sinais '
                   'identificados não representam diagnóstico médico ou '
           

In [48]:
print(
    result["audio_result"].get(
        "interpretation"
    )
)

O perfil vocal estimado é compatível com voz feminina. Foram observadas oscilações vocais compatíveis com instabilidade emocional ou tensão. Foram identificados sinais de hesitação na fala. A análise acústica identificou possível tensão vocal. A intensidade vocal reduzida pode estar associada a fadiga ou cansaço. A intensidade vocal identificada foi classificada como baixa. A elevada variação do pitch pode indicar agitação ou ansiedade. A presença de pausas frequentes pode indicar insegurança ou desconforto emocional. Os sinais identificados não representam diagnóstico médico ou psicológico, sendo utilizados apenas como apoio à triagem preventiva.


### CÉLULA 28 - RESULTADO MULTIMODAL

Nesta etapa é exibido o resultado da fusão multimodal entre áudio e vídeo.

In [49]:
pprint(result["multimodal_result"])

{'alert': False,
 'audio_risk_level': 'alto',
 'audio_score': 0.7,
 'evidences': ['person_detected',
               'face_not_visible',
               'voice_instability',
               'elevated_voice_tension',
               'speech_hesitation',
               'low_voice_energy'],
 'final_score': 0.5,
 'fusion_strategy': 'audio_60_video_40',
 'interpretation': ['A análise de áudio apresentou sinais relevantes de '
                    'possível ansiedade, agitação ou fadiga emocional.',
                    'A análise de vídeo apresentou baixo nível de risco '
                    'visual.'],
 'limitations': ['A análise multimodal é apenas apoio à triagem clínica.',
                 'O sistema não realiza diagnóstico médico, psicológico ou '
                 'psiquiátrico.',
                 'Expressões faciais indicam apenas emoções aparentes.',
                 'Sinais de áudio e vídeo devem ser interpretados como '
                 'evidências complementares.',
                 'A c

In [50]:
for item in result["multimodal_result"].get(
    "interpretation",
    []
):
    print("-", item)

- A análise de áudio apresentou sinais relevantes de possível ansiedade, agitação ou fadiga emocional.
- A análise de vídeo apresentou baixo nível de risco visual.


### CÉLULA 31 - RESPOSTA FINAL DO AGENTE

O modelo de linguagem gera uma resposta interpretativa utilizando:
- contexto multimodal;
- resultados da fusão;
- informações recuperadas via RAG.

A resposta possui finalidade exclusivamente educacional.

In [51]:
print(result["answer"])

 1. Resumo da avaliação: A análise multimodal identificou possíveis sinais de risco emocional, indicando possível ansiedade, agitação ou fadiga emocional.
2. Evidências observadas: Sinais relevantes identificados foram face_not_visible, voice_instability, elevated_voice_tension, speech_hesitation e low_voice_energy.
3. Nível de risco: O nível de risco é médio.
4. Recomendação: Recomenda-se acompanhamento e nova avaliação clínica caso os sinais persistam, se intensifiquem ou estejam associados a sofrimento emocional.
5. Limitações da análise:
   - A análise multimodal é apenas apoio à triagem clínica.
   - O sistema não realiza diagnóstico médico, psicológico ou psiquiátrico.
   - Expressões faciais indicam apenas emoções aparentes.
   - Sinais de áudio e vídeo devem ser interpretados como evidências complementares.
   - A confirmação clínica depende de avaliação profissional.


### CÉLULA 33 - CONCLUSÃO

O teste validou:
- integração multimodal;
- upload e download via Amazon S3;
- processamento de vídeo;
- processamento de áudio;
- análise vocal;
- fusão multimodal;
- geração automática de respostas;
- integração completa do LangGraph.

As análises possuem finalidade educacional e de apoio à triagem preventiva, não representando diagnóstico médico ou psicológico.

### CÉLULA 34 - LIMPEZA S3

Após os testes, os arquivos podem ser removidos do S3 para evitar armazenamento desnecessário.

In [ ]:
for key in [video_s3_key, audio_s3_key]:

    s3.delete_object(
        Bucket=bucket,
        Key=key
    )

    print("Removido do S3:", key)

### Observação sobre os testes com RAVDESS

A base RAVDESS utiliza frases padronizadas repetidas com diferentes emoções simuladas por atores. Por isso, neste teste, a transcrição textual pode não conter palavras-chave clínicas de alerta.

Nesse cenário, a análise de áudio depende principalmente de características acústicas, como pitch, pausas, energia vocal e variação da voz. Esses sinais são tratados como evidências complementares e não devem ser interpretados isoladamente como ansiedade, depressão ou violência.

No vídeo testado, a presença humana foi detectada, porém não houve detecção facial suficiente para análise emocional. Portanto, a modalidade visual não foi considerada como evidência emocional forte neste caso.

Esse resultado demonstra uma limitação importante do pipeline: bases sintéticas com falas padronizadas são úteis para validar o fluxo multimodal, mas exigem interpretação cautelosa dos sinais acústicos e visuais.